In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from langchain_community.document_loaders import TextLoader          # or PyPDFLoader for PDFs
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq


# 1. Load
docs = TextLoader("notes.txt", encoding="utf-8").load()

# 2. Split
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

# 3. Embed + 4. Store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5150.00it/s]


In [8]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

for d in retriever.invoke("What is the refund policy?"):
    print(d.page_content[:150], "\n---")

Company Handbook: Zentro Robotics (fictional)

Refund Policy
Customers may return any Zentro device within 45 days of delivery for a full refund.
Afte 
---
Warranty
Every Zentro Model Z1 vacuum robot includes a 3-year warranty on the motor
and a 1-year warranty on the battery. The warranty is void if the  
---
Shipping
Orders above 2000 rupees ship free within Kerala. Orders to other states take 5 to 7 business days.
International shipping is not available f 
---


In [9]:
model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context below. If the answer isn't there, say you don't know.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

print(rag_chain.invoke("What is the refund policy?"))

Customers can return any Zentro device within 45 days of delivery for a full refund. After 45 days, only store credit is offered, up to 60 percent of the purchase price. Devices with water damage are not eligible for refunds under any circumstances.
